# LayMesh 与 NumPy、pandas 和 Matplotlib 协作

这个示例面向已经会用 pandas 和 Matplotlib 的 Notebook 用户。运行前在当前 Notebook 内核环境执行 `pip install laymesh`，即可安装 LayMesh、NumPy、pandas 和 Matplotlib。依次运行所有单元格，将得到可预览的组合图、PDF、PNG 和可独立重渲染的 `.lay` 源码。

## 1. 准备数据与输出目录

NumPy 生成横坐标，pandas 管理结果。LayMesh Magic 的相对路径以当前工作目录为基准；这里把示例输出集中放在 `output/laymesh-jupyter`。 当前内核中重复执行全部单元格仍使用同一输出目录；Magic 的内联预览由 Rust CPU 渲染为双倍像素密度 JPEG（192 DPI、质量 95、4:4:4 采样；透明区域默认白底），与导出 DPI 分开。导出与预览共用一次原生编译；--preview-dpi、--preview-quality、--preview-background 可单独调整预览。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Keep the same output directory when running all cells again.
if "notebook_dir" not in globals():
    notebook_dir = Path.cwd()
output_dir = notebook_dir / "output" / "laymesh-jupyter"
output_dir.mkdir(parents=True, exist_ok=True)
os.chdir(output_dir)

x = np.linspace(0, 2 * np.pi, 100)
df = pd.DataFrame({"x": x, "y": np.sin(x)})
title = f"Peak: {df['y'].max():.2f}"
print(f"{len(df)} rows; output: {output_dir}")

## 2. 绘图并排版

把 DataFrame 交给 Matplotlib；`{{fig}}` 绑定 Figure，`{{title}}` 绑定已经算好的字符串。Magic 单元格的其余内容仍是普通 `.lay`。

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(df["x"], df["y"], color="#28618a")
ax.set_xlabel("x")
ax.set_ylabel("sin(x)")
plt.tight_layout()
plt.close(fig)

In [ ]:
%load_ext laymesh.ipython

In [ ]:
%%laymesh -o figure.pdf --save-source figure.lay
page = canvas(size=(160 mm, 100 mm), background="#ffff00")
chart = image(src={{fig}})
placed = page.add(chart,size=(130 mm, auto),
                  target=page.top_left, offset=(15 mm, 10 mm))
caption = text(content={{title}}, font_size=10 pt)
page.add(caption, target=placed.bottom_left, offset=(0 mm, 4 mm))

In [ ]:
assert Path("figure.pdf").is_file()
assert Path("figure.lay").is_file()
assert any(Path("figure.assets").glob("*.svg"))
print("PDF、独立 .lay 源码和 SVG 绘图素材已生成")

## 3. 热图的栅格回退

Matplotlib 热图的 SVG 含嵌入图像，当前 LayMesh SVG 导入器不接收这种节点。Magic 会提示回退，并按 `--plot-dpi` 输出绘图 PNG；`--dpi` 单独控制最终页面 PNG 的分辨率。

In [ ]:
heatmap, ax = plt.subplots(figsize=(3, 2))
ax.imshow(np.arange(16).reshape(4, 4))
ax.set_title("Heatmap")
heatmap.tight_layout()
plt.close(heatmap)

In [ ]:
%%laymesh -o heatmap.png --dpi 150 --plot-dpi 240 --save-source heatmap.lay
page = canvas(size=(100 mm, 70 mm), background="#ffffff")
chart = image(src={{heatmap}})
page.add(chart,size=(80 mm, auto), target=page.center, anchor=center)

In [ ]:
assert Path("heatmap.png").is_file()
assert any(Path("heatmap.assets").glob("*.png"))
print("热图 PNG 已生成；绘图素材使用 240 DPI")

## 4. 重渲染与练习

生成的 `.lay` 可以脱离 Notebook 工作。试着改变 Python 数据和 Figure 样式，再重新运行拼版单元格；生成素材会按内容哈希保存。注意占位符写成 `image(src={{fig}})`，不要在 `{{fig}}` 外再加引号。

In [ ]:
%laymesh figure.lay -o figure-again.svg

In [ ]:
# 练习参考：重新计算一个标题，然后重新运行上方的 %%laymesh 单元格。
new_title = f"Mean: {df['y'].mean():.3f}"
print(new_title)